# 19.1 模型分发与 OTA

演示分片校验、断点续传账本、二进制差分体积估算、灰度决策。

In [ ]:
import hashlib
import json
import math
import time
from dataclasses import dataclass, field
from typing import Optional
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    torch.manual_seed(42)
    print(f"PyTorch {torch.__version__}")
except Exception as e:
    torch = None
    print("torch unavailable:", e)

np.random.seed(42)

In [ ]:
def chunk_file(data: bytes, chunk_size=1024) -> list[dict]:
    out = []
    for i in range(0, len(data), chunk_size):
        part = data[i:i+chunk_size]
        out.append({"i": i//chunk_size, "sha": hashlib.sha256(part).hexdigest(), "bytes": part})
    return out


class DownloadLedger:
    def __init__(self, chunks):
        self.chunks = chunks
        self.done = set()

    def resume_from(self):
        for c in self.chunks:
            if c["i"] not in self.done:
                return c["i"]
        return None

    def accept(self, i, blob, sha):
        if hashlib.sha256(blob).hexdigest() != sha:
            return False
        self.done.add(i)
        return True


payload = b"FAKE-GGUF-" + b"x" * 5000
chunks = chunk_file(payload, 1200)
ledger = DownloadLedger(chunks)
# 模拟下到一半失败后恢复
for c in chunks[:2]:
    assert ledger.accept(c["i"], c["bytes"], c["sha"])
print("resume at", ledger.resume_from())
for c in chunks:
    if c["i"] not in ledger.done:
        assert ledger.accept(c["i"], c["bytes"], c["sha"])
print("complete", len(ledger.done), "/", len(chunks))

In [ ]:
def delta_ratio(old: bytes, new: bytes) -> float:
    # 极简估计：不同字节占比（真 OTA 用 bsdiff；这里只教学）
    n = max(len(old), len(new))
    old = old.ljust(n, b"\0"); new = new.ljust(n, b"\0")
    diff = sum(1 for a,b in zip(old,new) if a!=b)
    return diff / n


v1 = b"BASEWEIGHT" + b"A" * 2000
v2 = b"BASEWEIGHT" + b"A" * 1800 + b"B" * 200
v3 = b"BASEWEIGHT" + b"A" * 2000  # identical
print("v1->v2 changed_ratio", round(delta_ratio(v1,v2), 3), "prefer delta" if delta_ratio(v1,v2)<0.5 else "prefer full")
print("adapter-only OTA size << full model: e.g. 32MB lora vs 1200MB base")


def should_rollout(device_tier: str, battery: int, staged_pct: int, user_bucket: int) -> bool:
    if battery < 20:
        return False
    if device_tier == "low" and staged_pct < 50:
        return False
    return user_bucket < staged_pct

print("rollout?", should_rollout("mid", 55, 10, 7))
print("rollout low batt?", should_rollout("mid", 15, 10, 7))

## 小结

分片+哈希可续传；小改动用差分；能只更 LoRA 就不要推全量基座；灰度要看电量与设备档。